#  Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
###  What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

###  Bonus
- Improve routing
- Add logging
- Add more tools


## Calculator Tool



In [3]:
def calculator(expr):
    # basic calculator using eval, catches bad expressions
    try:
        ans = eval(expr)
        return str(ans)
    except:
        return "Error in calculation"


## Keyword Extractor Tool
Splits the text on spaces, keeps only words longer than 4 letters, drops duplicates and returns the first 5 as keywords. If something breaks it just falls back to an empty list.


In [4]:
def extract_keywords(text):
    # keep words longer than 4 letters, dedupe, take first 5
    try:
        w = text.split()
        kw = list(set([x.lower() for x in w if len(x) > 4]))
        return kw[:5]
    except:
        return []


## Bonus - 3rd Tool + Logging
Added a `text_stats` tool (word count + char count), set up a `logger` so we can see what intent each query got routed to, and a `TRAJECTORY_LOG` list that keeps every query along with its intent and response.


In [5]:
import logging

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger("single_agent")

TRAJECTORY_LOG = []  # keeps every query -> intent -> response

def text_stats(text):
    # word count + char count
    try:
        w = text.split()
        return {"word_count": len(w), "char_count": len(text)}
    except:
        return {"word_count": 0, "char_count": 0}


## Adding 5 More Tools

Adding reverse text, uppercase, palindrome check, temperature conversion and current date/time. That brings the agent up to 8 tools total.


In [6]:
# reverse text tool
def reverse_text(text):
    try:
        return text[::-1]
    except:
        return ""


# uppercase tool
def to_uppercase(text):
    try:
        return text.upper()
    except:
        return ""


# palindrome check, ignoring some filler words
def check_palindrome(text):
    try:
        skip = ["is", "a", "an", "the", "palindrome", "check", "if"]
        words = [w for w in text.lower().split() if w not in skip]
        joined = "".join(words)
        return joined == joined[::-1]
    except:
        return False


# celsius <-> fahrenheit
import re

def convert_temperature(query):
    try:
        m = re.search(r"[-+]?\d*\.?\d+", query)
        if not m:
            return "No number found"
        val = float(m.group())

        q = query.lower()
        c_pos = q.find("celsius")
        f_pos = q.find("fahrenheit")

        if c_pos == -1 and f_pos == -1:
            return "Please mention celsius or fahrenheit"

        # whichever unit shows up first in the text is treated as the source unit
        if f_pos != -1 and (c_pos == -1 or f_pos < c_pos):
            return str(round((val - 32) * 5 / 9, 2)) + " C"
        else:
            return str(round((val * 9 / 5) + 32, 2)) + " F"
    except:
        return "Error in conversion"


# current date and time
from datetime import datetime

def get_current_datetime():
    try:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    except:
        return ""


## Agent Logic

Basic conditional routing:
- query has "calculate" -> calculator
- query has "keywords" -> keyword extractor
- anything else -> general reply


## Agent Function (updated)
Initially it only matched exact words like "calculate" or "uppercase", so anything phrased differently just fell through to the general reply. Fixed this by giving every tool a small list of trigger words instead of one, and a plain math expression like "20 + 5" now works even without typing "calculate".


In [7]:
import re

# trigger words per intent
TRIGGERS = {
    "calculation": ["calculate", "compute", "solve"],
    "keywords": ["keyword", "keywords", "extract"],
    "stats": ["count", "stats", "statistics"],
    "reverse": ["reverse", "flip"],
    "uppercase": ["uppercase", "upper case", "capitalize", "caps"],
    "palindrome": ["palindrome"],
    "temperature": ["celsius", "fahrenheit", "temperature"],
    "datetime": ["date", "time", "today", "now"],
}

TOOLS = {
    "keywords": lambda q: extract_keywords(q),
    "stats": lambda q: text_stats(q),
    "reverse": lambda q: reverse_text(q),
    "uppercase": lambda q: to_uppercase(q),
    "palindrome": lambda q: check_palindrome(q),
    "temperature": lambda q: convert_temperature(q),
    "datetime": lambda q: get_current_datetime(),
}

def classify_intent(text):
    # a plain math expression like "20 + 5" counts as calculation even without the word calculate
    if re.search(r"\d+\s*[\+\-\*/]\s*\d+", text):
        return "calculation"
    for intent in TRIGGERS:
        if any(word in text for word in TRIGGERS[intent]):
            return intent
    return "general"

def agent(query):
    q_lower = query.lower()
    intent = classify_intent(q_lower)
    logger.info("Query: " + query + " | Routed to: " + intent)

    try:
        if intent == "calculation":
            expr = re.sub(r"[a-zA-Z?]", "", q_lower).strip()
            result = calculator(expr)
            if result == "Error in calculation":
                response = {"type": "error", "result": "Could not evaluate the expression."}
            else:
                response = {"type": "calculation", "result": result}
        elif intent == "general":
            response = {"type": "general", "result": "I don't have a tool for this. Try calculate, keywords, stats, reverse, uppercase, palindrome, temperature, or date/time."}
        else:
            response = {"type": intent, "result": TOOLS[intent](query)}
    except Exception as e:
        logger.error("Agent failed: " + str(e))
        response = {"type": "error", "result": "Something went wrong: " + str(e)}

    TRAJECTORY_LOG.append({"query": query, "intent": intent, "response": response})
    return response


## Output Format

Every response from the agent looks like this:
```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```


## Test Cases
Running the 3 sample queries (one per tool) through `agent()` to check the routing is working.


In [13]:
test_qs = [
    "Calculate 86 + 45",
    "Extract keywords from Baldwin was the emperor of the crusaders",
    "Who is baldwin?"
]

for q in test_qs:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)


Query: Calculate 86 + 45
Response: {'type': 'calculation', 'result': '131'}
--------------------------------------------------
Query: Extract keywords from Baldwin was the emperor of the crusaders
Response: {'type': 'keywords', 'result': ['baldwin', 'crusaders', 'extract', 'keywords', 'emperor']}
--------------------------------------------------
Query: Who is baldwin?
Response: {'type': 'general', 'result': "I don't have a tool for this. Try calculate, keywords, stats, reverse, uppercase, palindrome, temperature, or date/time."}
--------------------------------------------------


## Bonus: Stats Tool + Trajectory Log
Testing the stats tool, then printing `TRAJECTORY_LOG` as a table so we can see every query, what intent it got routed to, and what came back.


In [9]:
q = "Show word count stats for this sentence"
print("Query:", q)
print("Response:", agent(q))
print("-" * 50)

import pandas as pd
pd.DataFrame(TRAJECTORY_LOG)


Query: Show word count stats for this sentence
Response: {'type': 'stats', 'result': {'word_count': 7, 'char_count': 39}}
--------------------------------------------------


,query,intent,response
0,Calculate 20 + 5,calculation,"{'type': 'calculation', 'result': '25'}"
1,Extract keywords from Artificial Intelligence ...,keywords,"{'type': 'keywords', 'result': ['transforming'..."
2,What is machine learning?,general,"{'type': 'general', 'result': 'I don't have a ..."
3,Show word count stats for this sentence,stats,"{'type': 'stats', 'result': {'word_count': 7, ..."


## Testing the 5 New Tools
One query per new tool, just to confirm all 8 tools are working properly.


In [10]:
more_qs = [
    "Reverse this text",
    "Convert to uppercase please",
    "Is madam a palindrome",
    "Convert 100 celsius",
    "What is the current date and time"
]

for q in more_qs:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)


Query: Reverse this text
Response: {'type': 'reverse', 'result': 'txet siht esreveR'}
--------------------------------------------------
Query: Convert to uppercase please
Response: {'type': 'uppercase', 'result': 'CONVERT TO UPPERCASE PLEASE'}
--------------------------------------------------
Query: Is madam a palindrome
Response: {'type': 'palindrome', 'result': True}
--------------------------------------------------
Query: Convert 100 celsius
Response: {'type': 'temperature', 'result': '212.0 F'}
--------------------------------------------------
Query: What is the current date and time
Response: {'type': 'datetime', 'result': '2026-07-12 15:06:18'}
--------------------------------------------------


## Interactive Mode
Type a query and get a live response from the agent, keep going until you type `exit`. Wrapped in try/except so it exits on its own if there's no input available (like during an automated run).


In [11]:
# interactive mode
while True:
    try:
        q = input("Enter query (type 'exit' to stop): ")
    except:
        print("(No interactive input available in this run — exiting loop.)")
        break
    if q.lower() == "exit":
        break
    print("Response:", agent(q))


Enter query (type 'exit' to stop): what is 14 +90
Response: {'type': 'calculation', 'result': '104'}
Enter query (type 'exit' to stop): reverse sanchit
Response: {'type': 'reverse', 'result': 'tihcnas esrever'}
Enter query (type 'exit' to stop): is sanchit a palindrome?
Response: {'type': 'palindrome', 'result': False}
Enter query (type 'exit' to stop): capitalize sanchit
Response: {'type': 'uppercase', 'result': 'CAPITALIZE SANCHIT'}
Enter query (type 'exit' to stop): keywords in machine learning
Response: {'type': 'keywords', 'result': ['learning', 'keywords', 'machine']}
Enter query (type 'exit' to stop): word count in my name is sanchit
Response: {'type': 'stats', 'result': {'word_count': 7, 'char_count': 32}}
Enter query (type 'exit' to stop): convert 100 farenheit into celsius
Response: {'type': 'temperature', 'result': '212.0 F'}
Enter query (type 'exit' to stop): exit


## Observation

Testing the agent showed it picks the right tool based on which words show up in the query - "calculate" goes to the calculator, "uppercase" goes to the case converter, and so on. When nothing matches, it just falls back to a general reply instead of crashing.

I also threw some bad input at it, like an invalid math expression, and it handled that fine too - returned an error message instead of breaking. The logs are useful here since they show exactly which tool each query got sent to.

One limitation I noticed: routing only works if the query has the exact trigger word in it. So something like "make this text loud" won't be understood as an uppercase request.


## Conclusion

For this assignment I built a single-agent assistant that reads a query, figures out which tool it needs, runs it, and returns a clean structured result. It covers math, keyword extraction and general queries, with error handling so bad input doesn't crash it.

For the bonus, I added logging so every query gets tracked, and went past the required 2 tools to build 8 in total. The routing logic is simple enough that adding a new tool later is just a matter of adding one more entry to TRIGGERS and TOOLS.
